# Answer Generation
Implements the **Phase 2 answer step** of Vectorless RAG.

This notebook reuses the retrieval pipeline from `retreivalPDF.ipynb` (tree
traversal → candidate pages → re-ranking → page fetch) and adds the final
step: asking the LLM to write a grounded answer using **only** the retrieved
page text.

Pipeline for a single user query:

```
query
  -> traverse_tree()        (Root -> Chapter -> Section, LLM-guided)
  -> get_candidate_pages()  (pages in winning section)
  -> rank_pages()           (LLM re-ranks by relevance)
  -> fetch_pages_content()  (full text for top-k pages)
  -> generate_answer()      (LLM answers from that text only)
```

**Run order:** `pageMetadata.ipynb` → `treeBuilder.ipynb` → (`retreivalPDF.ipynb`) → `answerGeneration.ipynb`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 1. Imports & Connection

In [2]:
import json
import re
import time

from src.config.db import get_connection
from src.config.llm import llm

conn = get_connection()

## 2. Set Document

In [3]:
document_id = "DOC000001"

## 3. Retrieval Pipeline (reused from `retreivalPDF.ipynb`)
Copied here so this notebook can run standalone. See `retreivalPDF.ipynb` for
the full walkthrough/explanation of each step.

In [4]:
class TreeNode:
    """Lightweight wrapper around a tree dict node."""

    def __init__(self, data: dict, parent=None):
        self.data = data
        self.parent = parent
        self.children = []

    @property
    def title(self):
        return self.data.get("title", "")

    @property
    def summary(self):
        return self.data.get("summary", "")

    @property
    def path(self):
        return self.data.get("path", "")

    @property
    def type(self):
        return self.data.get("type", "root")

    def __repr__(self):
        return f"TreeNode(path={self.path!r}, title={self.title!r})"


def build_tree_nodes(tree_dict: dict, parent=None) -> "TreeNode":
    node = TreeNode(tree_dict, parent)
    for child_dict in tree_dict.get("children", []):
        child_node = build_tree_nodes(child_dict, parent=node)
        node.children.append(child_node)
    return node


def load_tree_from_db(conn, document_id: str) -> "TreeNode":
    with conn.cursor() as cur:
        cur.execute(
            'SELECT "treeJson" FROM "Tree" WHERE "documentId" = %s',
            (document_id,),
        )
        row = cur.fetchone()

    if row is None:
        raise ValueError(f"No tree found for documentId={document_id!r}")

    tree_dict = row[0]
    if isinstance(tree_dict, str):
        tree_dict = json.loads(tree_dict)

    return build_tree_nodes(tree_dict)


def _parse_choice(response_text: str, num_children: int) -> int:
    match = re.search(r"\d+", response_text)
    if match:
        choice = int(match.group())
        if 1 <= choice <= num_children:
            return choice
    return 1


def choose_child(query: str, current: "TreeNode", llm) -> int:
    children = current.children

    prompt = f"""You are navigating a document tree to answer a user's question.

User Query:
{query}

Current Node ({current.type}):
{current.title}

Available Children:
"""
    for i, child in enumerate(children):
        prompt += f"""
{i + 1}. [{child.type}] {child.title}
   Summary: {child.summary}
"""
    prompt += """
Which child is most likely to contain the answer?
Return ONLY the number of the child. No explanation, no punctuation.
"""

    response = llm.invoke(prompt).content.strip()
    return _parse_choice(response, len(children))


def traverse_tree(query: str, root: "TreeNode", llm) -> dict:
    current = root
    path = [root]

    while current.children:
        choice = choose_child(query, current, llm)
        current = current.children[choice - 1]
        path.append(current)

    return {"path": path, "leaf": current}


def get_candidate_pages(conn, leaf: "TreeNode") -> list:
    page_ids = leaf.data.get("pageIds", [])
    if not page_ids:
        return []

    placeholders = ",".join(["%s"] * len(page_ids))
    with conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT "pageNumber", metadata
            FROM "Page"
            WHERE id IN ({placeholders})
            ORDER BY "pageNumber"
            ''',
            page_ids,
        )
        rows = cur.fetchall()

    candidates = []
    for page_number, metadata in rows:
        meta = metadata or {}
        candidates.append({
            "pageNumber": page_number,
            "title": meta.get("title", ""),
            "summary": meta.get("summary", ""),
            "keywords": meta.get("keywords", []),
        })
    return candidates


def rank_pages(query: str, candidates: list, llm) -> list:
    if not candidates:
        return []

    valid_pages = {c["pageNumber"] for c in candidates}

    prompt = f"""User Question:
{query}

Candidate Pages:
"""
    for page in candidates:
        prompt += f"""
Page {page['pageNumber']}
Title: {page['title']}
Summary: {page['summary']}
Keywords: {", ".join(page['keywords'])}
"""
    prompt += """
Return ONLY the page numbers, ranked from most relevant to least relevant,
one per line. No explanation, no extra text.

Example:
112
114
113
"""

    response = llm.invoke(prompt).content

    ranked, seen = [], set()
    for line in response.splitlines():
        for d in re.findall(r"\d+", line.strip()):
            page_num = int(d)
            if page_num in valid_pages and page_num not in seen:
                ranked.append(page_num)
                seen.add(page_num)

    for c in candidates:
        if c["pageNumber"] not in seen:
            ranked.append(c["pageNumber"])
            seen.add(c["pageNumber"])

    return ranked


def fetch_pages_content(conn, document_id: str, page_numbers: list) -> dict:
    if not page_numbers:
        return {}

    placeholders = ",".join(["%s"] * len(page_numbers))
    with conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT "pageNumber", content
            FROM "Page"
            WHERE "documentId" = %s
              AND "pageNumber" IN ({placeholders})
            ''',
            (document_id, *page_numbers),
        )
        rows = cur.fetchall()

    content_by_page = {page_number: content for page_number, content in rows}
    return {p: content_by_page[p] for p in page_numbers if p in content_by_page}


def retrieve(query: str, document_id: str, conn, llm, top_k: int = 3) -> dict:
    """End-to-end retrieval: tree traversal -> candidate pages -> re-rank -> fetch content."""
    tree_root = load_tree_from_db(conn, document_id)
    traversal = traverse_tree(query, tree_root, llm)
    leaf = traversal["leaf"]

    candidates = get_candidate_pages(conn, leaf)
    ranked_pages = rank_pages(query, candidates, llm)
    top_pages = ranked_pages[:top_k]
    pages_content = fetch_pages_content(conn, document_id, top_pages)

    return {
        "tree_path": [
            {"type": n.type, "title": n.title, "path": n.path}
            for n in traversal["path"]
        ],
        "leaf": leaf.data,
        "candidates": candidates,
        "ranked_pages": ranked_pages,
        "top_pages": top_pages,
        "pages_content": pages_content,
    }

## 4. Answer Generation Prompt
The LLM is instructed to answer **only** from the provided pages, cite the
page number(s) it drew from, and explicitly say so if the answer isn't
present — this keeps the pipeline grounded and avoids hallucination.

In [5]:
ANSWER_PROMPT = """You are answering a question using ONLY the context pages below.
This is a Vectorless RAG system — there is no other source of truth.

Rules:
1. Answer using only the provided context. Do not use outside knowledge.
2. If the answer is not contained in the context, say so explicitly —
   do not guess or make anything up.
3. When you use information from a page, cite it inline as (Page N).
4. Be concise and directly answer the question first, then add supporting detail.

Question:
{query}

Context:
{context}

Answer:
"""

## 5. Generate Answer Function

In [6]:
def build_context(pages_content: dict) -> str:
    """Format retrieved pages into a single context block, tagged by page number."""
    blocks = []
    for page_number, text in pages_content.items():
        blocks.append(f"========== Page {page_number} ==========\n{text}")
    return "\n\n".join(blocks)


def generate_answer(query: str, pages_content: dict, llm) -> str:
    """Ask the LLM to answer the query using only the given page content."""
    if not pages_content:
        return "I couldn't find any relevant pages in the document to answer this question."

    context = build_context(pages_content)
    prompt = ANSWER_PROMPT.format(query=query, context=context)

    response = llm.invoke(prompt)
    return response.content.strip()

## 6. Full Answer Pipeline (Retrieve + Generate)
Ties retrieval and generation together into the single function the rest of
the app (API route / evaluation notebook) should call.

In [7]:
def answer_query(query: str, document_id: str, conn, llm, top_k: int = 3) -> dict:
    """
    Full Phase 2 pipeline: retrieve relevant pages via tree traversal + re-ranking,
    then generate a grounded answer from their content.

    Returns:
      {
        "query": str,
        "answer": str,
        "sources": [page_number, ...],   # pages actually used for the answer
        "tree_path": [...],              # traversal path for debugging/explainability
        "latency_seconds": float,
      }
    """
    start = time.time()

    retrieval = retrieve(query, document_id, conn, llm, top_k=top_k)
    answer = generate_answer(query, retrieval["pages_content"], llm)

    elapsed = time.time() - start

    return {
        "query": query,
        "answer": answer,
        "sources": retrieval["top_pages"],
        "tree_path": retrieval["tree_path"],
        "latency_seconds": round(elapsed, 2),
    }

## 7. Optional — Log Queries Locally
Keeps a simple append-only log of every query/answer for later inspection or
evaluation, without assuming a specific DB table exists yet. If you add a
`QueryLog` table to the schema later, swap this for an `INSERT`.

Commented out for now — uncomment `log_query(...)` calls below if you want
to turn this on.

In [8]:
# import os
#
# LOG_PATH = "storage/query_logs.jsonl"
#
#
# def log_query(document_id: str, result: dict):
#     os.makedirs("storage", exist_ok=True)
#     record = {"documentId": document_id, **result}
#     with open(LOG_PATH, "a", encoding="utf-8") as f:
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")

## 8. Demo

In [9]:
query = "What Summarize Monitoring Financial Vulnerabilities?"

result = answer_query(query, document_id, conn, llm, top_k=3)

print("Query:", result["query"])
print()
print("Answer:")
print(result["answer"])
print()
print("Sources (pages):", result["sources"])
print("Latency:", result["latency_seconds"], "s")

Query: What Summarize Monitoring Financial Vulnerabilities?

Answer:
The Federal Reserve Board summarizes monitoring financial vulnerabilities through its Financial Stability Report, which assesses the resilience of the U.S. financial system and presents the Board's current assessment of financial system vulnerabilities (Page 23). The report aims to promote public understanding and increase transparency and accountability. The Board monitors a set of vulnerabilities, including asset valuation pressures, borrowing by households and businesses, leverage in the financial sector, and funding risk, to inform discussions on policies to promote financial stability (Page 23).

Sources (pages): [23, 24]
Latency: 0.96 s


## 9. Interactive Query (optional, manual testing)
Run this cell to ask a question interactively from the notebook.

In [10]:
question = input("Enter your question: ")

result = answer_query(question, document_id, conn, llm, top_k=3)

print("\nAnswer:")
print(result["answer"])
print("\nSources (pages):", result["sources"])


Answer:
Monitoring Financial Vulnerabilities involves assessing the level and configuration of vulnerabilities that can affect the financial system's resilience to adverse shocks (Page 23). The Federal Reserve maintains a flexible, forward-looking financial stability monitoring program that focuses on vulnerabilities such as asset valuation pressures, borrowing by households and businesses, leverage in the financial sector, and funding risk (Page 23). The program informs discussions on policies to promote financial stability and increases transparency and accountability through publications like the Financial Stability Report (Page 23). 

Supporting details include tracking a broad range of measures, such as price volatility, underwriting standards, and investor flows, to analyze asset valuation pressures (Page 23), and monitoring borrowing by households and businesses, which can contribute to past financial crises (Page 24).

Sources (pages): [24, 23]
